In [5]:
# --- Run ---
import datetime

run_timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

In [6]:
import subprocess, sys, os

"""
This output is:
 1. pubs.txt - Paper list for CWTS tool
    Columns: int_id  paper identifier for cwts, core_pub (always 1 as not using core feature, but it has to be in)

 2. cit_links.txt - Citation network edges
    Columns: 
    int_id1 - citing paper, 
    int_id2 - cited paper, 
    weight - citation strength 0-2 higher= stronger
    Note: Each edge appears twice (A→B and B→A) for undirected format
        paper 5 cites Paper 12  →  row: 5, 12, 0.85
        Paper 12 cites Paper 5  →  row: 12, 5, 0.85  (same edge, reversed)

 3. pub_metadata.txt - Paper details lookup table
    Columns: int_id, pub_id, is_frontiers, journal, date, title
    - int_id: sequential CWTS ID (joins to classification.txt)
    - pub_id: airak PublicationId (joins to BigQuery tables)
 JOIN KEY: int_id links all files together, this is cwts identifier
"""
print("ere")
env = os.environ.copy()
print("ere")
env["START_YEAR"] = "2023"
env["END_YEAR"] = "2026"
env["NETWORK_MODE"] = "full"
env["run_timestamp"] = run_timestamp
print("ere")
result = subprocess.run(
    [sys.executable, "cwts_export.py"], capture_output=True, text=True, env=env
)
print("ere")
print(result.stdout)
print(result.stderr)
print("last")

ere
ere
ere
ere

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
2026-06-18 13:03:11,853 [INFO] ============================================================
2026-06-18 13:03:11,853 [INFO] Configuration
2026-06-18 13:03:11,853 [INFO] ============================================================
2026-06-18 13:03:11,853 [INFO]   BQ_PROJECT              : ocean-tech-adv-analytics-p-usr
2026-06-18 13:03:11,853 [INFO]   AIRAK_DATASET           : ocean-breeze-tier-1.airak
2026-06-18 13:03:11,853 [INFO]   NETWORK_MODE            : full
2026-06-18 13:03:11,854 [INFO]   START_YEAR              : 2023
2026-06-18 13:03:11,854 [INFO]   END_YEAR                : 2026
2026-06-18 13:03:11,854 [INFO]   TOP_N_JOURNALS          : 5
2026-06-18 13:03:11,855 [INFO]   JOURNAL_IDS_OVERRIDE    : (none)
2026-06-18 13:03:11,

In [17]:
import subprocess
import datetime
import os
import pandas as pd

# --- Parameters ---
params = {
    "largest_component_only": "true",
    "iterations": "100",
    "micro_resolution": "5e-4",
    "micro_min_cluster_size": "1000",
    "meso_resolution": "5e-6",
    "meso_min_cluster_size": "5000",
    "macro_resolution": "1e-6",
    "macro_min_cluster_size": "20000",
}

input_files = {
    "pubs": "cwts_output/pubs.txt",
    "cit_links": "cwts_output/cit_links.txt",
    "output": "cwts_output/classification.txt",
    "jar": "publicationclassification.jar",
}


result = subprocess.run(
    [
        "java",
        "-cp",
        input_files["jar"],
        "nl.cwts.publicationclassification.run.PublicationClassificationCreator",
        input_files["pubs"],
        input_files["cit_links"],
        input_files["output"],
        params["largest_component_only"],
        params["iterations"],
        params["micro_resolution"],
        params["micro_min_cluster_size"],
        params["meso_resolution"],
        params["meso_min_cluster_size"],
        params["macro_resolution"],
        params["macro_min_cluster_size"],
    ],
    capture_output=True,
    text=True,
)

# --- Log ---
os.makedirs("logs", exist_ok=True)
log_path = f"logs/cwts_run_{run_timestamp}.log"

with open(log_path, "w") as f:
    f.write(f"CWTS Publication Classification Run\n")
    f.write(f"{'='*50}\n")
    f.write(f"Timestamp : {run_timestamp}\n\n")

    f.write(f"Input Files\n{'-'*30}\n")
    for k, v in input_files.items():
        f.write(f"  {k:<20}: {v}\n")

    f.write(f"\nParameters\n{'-'*30}\n")
    for k, v in params.items():
        f.write(f"  {k:<26}: {v}\n")

    f.write(f"\nReturn Code: {result.returncode}\n")

    f.write(f"\nSTDOUT\n{'-'*30}\n")
    f.write(result.stdout or "(empty)\n")

    f.write(f"\nSTDERR\n{'-'*30}\n")
    f.write(result.stderr or "(empty)\n")

print(f"Log written to: {log_path}")
print(result.stdout)
if result.stderr:
    print(result.stderr)

# Load classification.txt
classification = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["int_id", "micro", "meso", "macro"],
)

# Upload to BigQuery
BQ_DEST_PROJECT = "ocean-tech-adv-analytics-c-tfs"
BQ_DEST_DATASET = "scope_drift_raw"


# classification.to_gbq(
#     f"{dataset}.classification_raw_{run_timestamp}",
#     project_id=project,
#     if_exists="replace",
# )

# Upload to BigQuery

classification.to_gbq(
    f"{BQ_DEST_DATASET}.classification_raw_{run_timestamp}",
    project_id=BQ_DEST_PROJECT,
    if_exists="replace",
)
print(f"  → BigQuery: {BQ_DEST_DATASET}.classification_raw_{run_timestamp}")

Log written to: logs/cwts_run_20260618_130306.log
PublicationClassificationCreator version 1.1.0
By Nees Jan van Eck
Centre for Science and Technology Studies (CWTS), Leiden University

Reading citation network from file... Finished!
Reading citation network from file took 0h 0m 26s.
Citation network:
	Number of publications: 3943071
	Number of citation links: 36537120
	Total publication weight: 3943071
	Total citation link weight: 10948391

Identifying largest connected component in citation network... Finished!
Identifying largest connected component in citation network took 0h 0m 3s.
Largest connected component:
	Number of publications: 3844339
	Number of citation links: 36478421
	Total publication weight: 3844339
	Total citation link weight: 10918126

Creating publication classification...
	Clustering algorithm: Leiden algorithm
	Number of iterations: 100
	Random seed: 0

Adding micro-level classification...
Creating clustering... Finished! 124673 clusters created.
Reassigning smal

C:\Users\sophie.wilson\AppData\Local\Temp\ipykernel_8988\3056501446.py:99: FutureWarning: to_gbq is deprecated and will be removed in a future version. Please use pandas_gbq.to_gbq instead: https://pandas-gbq.readthedocs.io/en/latest/api.html#pandas_gbq.to_gbq
  classification.to_gbq(
100%|██████████| 1/1 [00:00<00:00, 499.62it/s]

  → BigQuery: scope_drift_raw.classification_raw_20260618_130306


In [ ]:
import pandas as pd


df = pd.read_csv(
    "cwts_output/classification.txt",
    sep="\t",
    header=None,
    names=["pub_no", "micro", "meso", "macro"],
)


print(f"Total classified: {len(df):,}")


for level in ["micro", "meso", "macro"]:

    vc = df[level].value_counts()

    print(f"\n{level.upper()}: {len(vc):,} clusters")

    print(f"  Largest : {vc.iloc[0]:,} ({vc.iloc[0]/len(df)*100:.1f}%)")

    print(f"  Smallest: {vc.iloc[-1]:,}")

    print(f"  Median  : {vc.median():.0f}")

Total classified: 3,844,339

MICRO: 1,818 clusters
  Largest : 16,117 (0.4%)
  Smallest: 1,000
  Median  : 1772

MESO: 183 clusters
  Largest : 298,499 (7.8%)
  Smallest: 5,076
  Median  : 12444

MACRO: 24 clusters
  Largest : 1,237,462 (32.2%)
  Smallest: 20,510
  Median  : 92906


### Labelling with GPT

In [1]:
import label_clusters

# Run the script
label_clusters.main()

C:\Users\sophie.wilson\AppData\Roaming\Python\Python310\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


Loading classification...
Loading metadata...
Loading citation links...
Calculating citation counts...
Merged: 3,844,339 publications

--- Labelling meso (183 clusters) ---
  [1/183] cluster 0 (298,499 papers) → Cancer Research
  [2/183] cluster 1 (94,639 papers) → Energy Storage
  [3/183] cluster 2 (82,959 papers) → Sustainable Development
  [4/183] cluster 3 (82,464 papers) → Bone Regeneration
  [5/183] cluster 4 (81,039 papers) → Gut Microbiome Research
  [6/183] cluster 5 (76,233 papers) → Mental Health Research
  [7/183] cluster 6 (71,230 papers) → Medical Imaging
  [8/183] cluster 7 (69,415 papers) → Artificial Intelligence Education
  [9/183] cluster 8 (66,199 papers) → Neurodegenerative Diseases
  [10/183] cluster 9 (61,830 papers) → Biosensors and Diagnostics
  [11/183] cluster 10 (61,446 papers) → COVID-19 Research
  [12/183] cluster 11 (57,840 papers) → Food Science
  [13/183] cluster 12 (56,743 papers) → Antimicrobial Resistance
  [14/183] cluster 13 (52,484 papers) → Neuro

## Labelling with taxonomy

In [2]:
import taxonomy_naming

taxonomy_naming.RUN_TIMESTAMP = run_timestamp
taxonomy_naming.CLUSTER_LEVEL = "macro"  # or "meso", "micro"
taxonomy_naming.main()

NameError: name 'run_timestamp' is not defined

### looking at scope

In [ ]:
import importlib
import journal_scope
from pathlib import Path

# Override config variables
journal_scope.SCOPE_LEVEL = "macro"
journal_scope.SCOPE_THRESHOLD = 0.80
journal_scope.MIN_PAPERS = 50
journal_scope.USE_GPT = False
journal_scope.OUTPUT_DIR = Path("cwts_output")

journal_scope.TARGET_JOURNALS = [
    "Frontiers in Immunology",
    "Frontiers in Public Health",
    "Frontiers in Medicine",
    "Frontiers in Oncology",
    "Frontiers in Psychology",
]

# Run
journal_scope.main()

Loading classification...
Loading metadata (journal + title)...
Merged: 67,823 publications across 5 journals
After MIN_PAPERS=50 filter: 5 journals, 67,823 papers

Computing scope at 'macro' level (threshold=80%)...
  Frontiers in Immunology                  n=19,501  core_clusters=  3  OOS=17.4%
  Frontiers in Medicine                    n=10,614  core_clusters=  9  OOS=18.4%
  Frontiers in Oncology                    n=12,794  core_clusters=  2  OOS=19.2%
  Frontiers in Psychology                  n=11,438  core_clusters=  3  OOS=19.9%
  Frontiers in Public Health               n=13,476  core_clusters=  7  OOS=17.7%

Saved journal scope → cwts_output\journal_scope.csv  (5 journals)
Saved paper flags   → cwts_output\paper_oos_flags.csv  (67,823 papers)

── Summary ──────────────────────────────────────────────────────
Journals analysed:        5
Mean OOS rate:            18.5%
Median OOS rate:          18.4%
Journals with OOS > 20%:  0
Journals with OOS > 40%:  0

Top 10 highest OOS 

### Generate Dashboard

In [1]:
import subprocess
import sys
import os

# --- Config ---
CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# Build the scope dashboard from local CWTS files
# Uses: cwts_output/classification.txt, cwts_output/pub_metadata.txt, cwts_output/cit_links.txt
# Outputs: output/scope_dashboard.html

env = os.environ.copy()
env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

result = subprocess.run(
    [sys.executable, "scripts/build_dashboard_from_cwts.py"],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)
else:
    print(f"\nDashboard ready: output/scope_dashboard.html")



Dashboard ready: output/scope_dashboard.html


In [1]:
import subprocess
import sys
import os

# --- Config ---
CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# Build the drift dashboard (JSD trends, heatmap, entropy changes)
# Compares current cluster distribution vs baseline (2018-2020)
# Outputs: output/drift_dashboard.html

env = os.environ.copy()
env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

result = subprocess.run(
    [sys.executable, "scripts/build_drift_dashboard_from_cwts.py"],
    capture_output=True,
    text=True,
    env=env,
)

print(result.stdout)
if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)
else:
    print(f"\nDashboard ready: output/drift_dashboard.html")



Dashboard ready: output/drift_dashboard.html


In [ ]:
import subprocess
import sys
import os

# # --- Config ---
# CLUSTER_LEVEL = "macro"  # Options: micro, meso, macro

# # Build the cluster bubble dashboard
# # Shows clusters as bubbles positioned by citation relationships
# # Outputs: output/cluster_bubbles.html

# env = os.environ.copy()
# env["CLUSTER_LEVEL"] = CLUSTER_LEVEL

# result = subprocess.run(
#     [sys.executable, "scripts/build_cluster_bubbles_from_cwts.py"],
#     capture_output=True,
#     text=True,
#     env=env,
# )

# print(result.stdout)
# if result.returncode != 0:
#     print("ERRORS:")
#     print(result.stderr)
# else:
#     print(f"\nDashboard ready: output/cluster_bubbles.html")

In [3]:
import subprocess
import sys

# Build the clusters hierarchy dashboard
# Shows macro/meso/micro clusters with GPT labels
# Outputs: output/clusters.html

result = subprocess.run(
    [sys.executable, "scripts/build_clusters_from_cwts.py"],
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode != 0:
    print("ERRORS:")
    print(result.stderr)
else:
    print(f"\nDashboard ready: output/clusters.html")



Dashboard ready: output/clusters.html
